In [9]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn glob2 pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: c:\Users\Dat\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import glob
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

# ==========================================
# LE THANH DAT (TASK 2.2)
# 1. TỔNG HỢP VÀ DỌN DẸP DỮ LIỆU 
# ==========================================

path = '../data/*.csv'
files = glob.glob(path)

# Nếu đã có file pkl rồi thì load luôn, không cần đọc lại 8 CSV
if os.path.exists('../models/cleaned_data.pkl'):
    print("Tìm thấy cleaned_data.pkl, đang load...")
    df = joblib.load('../models/cleaned_data.pkl')
    print("Load xong!")

elif not files:
    print("LỖI: Không tìm thấy file CSV nào trong thư mục data!")

else:
    # Gộp tất cả file thành 1 DataFrame lớn
    print(f"Đang gộp {len(files)} file dữ liệu...")
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    # Dọn dẹp khoảng trắng trong tên cột
    df.columns = df.columns.str.strip()

    # Xử lý giá trị vô hạn (inf) phát sinh do lỗi log mạng
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Điền các giá trị trống bằng trung vị (median) của từng cột số
    df.fillna(df.median(numeric_only=True), inplace=True)

    # Loại bỏ dữ liệu trùng lặp để tránh model bị học vẹt
    df.drop_duplicates(inplace=True)

    # Tối ưu RAM bằng cách chuyển float64 về float32
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')

    # Lưu lại để lần sau không cần đọc lại 8 CSV
    joblib.dump(df, '../models/cleaned_data.pkl')
    print("Hoàn thành sơ chế dữ liệu và đã lưu cleaned_data.pkl!")

Tìm thấy cleaned_data.pkl, đang load...
Load xong!


In [2]:
# ==========================================
# NGUYEN QUOC DAT 
# 2.3 XỬ LÝ MẤT CÂN BẰNG DỮ LIỆU
# ==========================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import joblib

# Load df nếu chưa có
if 'df' not in dir():
    df = joblib.load('../models/cleaned_data.pkl')

# Encode Label
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
print("Các nhãn:", list(le.classes_))

# Tách X (78 features), y
X = df.drop('Label', axis=1)
y = df['Label']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Xem phân phối trước
print("\nTrước khi balance:")
print(pd.Series(y_train).value_counts())

# SMOTE + RandomUnderSampler
majority_count = pd.Series(y_train).value_counts().max()
threshold = int(majority_count * 0.1)

sampling_strategy_smote = {
    cls: max(count, threshold)
    for cls, count in pd.Series(y_train).value_counts().items()
    if count < majority_count
}

pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=sampling_strategy_smote, random_state=42)),
    ('under', RandomUnderSampler(sampling_strategy='majority', random_state=42))
])

X_train, y_train = pipeline.fit_resample(X_train, y_train)

print("\nSau khi balance:")
print(pd.Series(y_train).value_counts())

# Lưu lại
joblib.dump((X_train, X_test, y_train, y_test), '../models/balanced_data.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')
print("\nĐã lưu balanced_data.pkl!")

Các nhãn: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']

Trước khi balance:
Label
0     1677187
4      138279
2      102413
10      72655
3        8229
7        4746
6        4308
5        4182
11       2575
1        1562
12       1176
14        522
9          29
13         17
8           9
Name: count, dtype: int64

Sau khi balance:
Label
0     167718
1     167718
2     167718
3     167718
4     167718
5     167718
6     167718
7     167718
8     167718
9     167718
10    167718
11    167718
12    167718
13    167718
14    167718
Name: count, dtype: int64

Đã lưu balanced_data.pkl!


In [ ]:
# ==========================================
# LE THANH DAT (TASK 2.5)
# 2. HUẤN LUYỆN MÔ HÌNH BASELINE 
# ==========================================

# Mã hóa nhãn (Chuyển tên cuộc tấn công thành số)
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

# Chia đặc trưng (X) và nhãn mục tiêu (y)
X = df.drop('Label', axis=1)
y = df['Label']

# Chia tập Train (80%) và Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Chuẩn hóa dữ liệu (Scaling) - Cực kỳ quan trọng cho Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Huấn luyện mô hình Logistic Regression
# Tăng max_iter để đảm bảo mô hình hội tụ tốt hơn
print("Đang huấn luyện mô hình Logistic Regression (có thể mất vài phút)...")
model = LogisticRegression(max_iter=500, solver='lbfgs')
model.fit(X_train_scaled, y_train)

# ==========================================
# 3. ĐÁNH GIÁ VÀ LƯU TRỮ (OUTPUT)
# ==========================================

# Tính toán độ chính xác tổng quát
accuracy = model.score(X_test_scaled, y_test)
print(f"\n--- KẾT QUẢ ---")
print(f"Độ chính xác tổng thể: {accuracy:.4f}")

# Dự đoán và in báo cáo chi tiết (Precision, Recall, F1-score)
y_pred = model.predict(X_test_scaled)
print("\nBáo cáo chi tiết cho từng loại tấn công:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Tạo thư mục và lưu trữ model đã học
joblib.dump(model, '../models/logistic_regression.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("\nĐã lưu 'logistic_regression.pkl' và 'scaler.pkl' vào thư mục models.")